In [3]:
# ================================
# 1. Import Libraries
# ================================
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# ML Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

# Download NLTK resources (Kaggle safe)
nltk.download('punkt')
nltk.download('stopwords')

# ================================
# 2. Load Dataset
# ================================
# Update path if needed
df = pd.read_csv('/kaggle/input/datasets/utkarshx27/consumer-complaint/complaints.csv')

# Preview
print("Dataset Shape:", df.shape)
df.head()

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/tmp/ipykernel_57/1250959641.py:25: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/kaggle/input/datasets/utkarshx27/consumer-complaint/complaints.csv')


Dataset Shape: (3585952, 18)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2023-04-27,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"EQUIFAX, INC.",GA,30308.0,NaN,Other,Web,2023-04-27,In progress,Yes,NaN,6896105
1,2023-04-27,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"EQUIFAX, INC.",PA,19145.0,NaN,NaN,Web,2023-04-27,In progress,Yes,NaN,6896026
2,2023-04-27,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,Their investigation did not fix an error on yo...,NaN,NaN,"EQUIFAX, INC.",PA,19111.0,NaN,NaN,Web,2023-04-27,In progress,Yes,NaN,6896030
3,2023-04-27,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"EQUIFAX, INC.",TX,78725.0,NaN,NaN,Web,2023-04-27,In progress,Yes,NaN,6896035
4,2023-04-27,"Credit reporting, credit repair services, or o...",Credit reporting,Improper use of your report,Reporting company used your report improperly,NaN,NaN,"EQUIFAX, INC.",NY,11233.0,NaN,NaN,Web,2023-04-27,In progress,Yes,NaN,6896060


In [4]:
# ================================
# 3. Basic Data Cleaning
# ================================

# Keep only relevant columns (modify if needed)
columns_needed = [
    'Consumer complaint narrative',
    'Product',
    'Issue',
    'Date received',
    'Date sent to company'
]

df = df[columns_needed]

# Drop missing complaint text
df = df.dropna(subset=['Consumer complaint narrative'])

df.reset_index(drop=True, inplace=True)

print("After cleaning:", df.shape)

After cleaning: (1292265, 5)


In [5]:
# ================================
# 4. Text Preprocessing (NLP)
# ================================

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase
    text = text.lower()
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # Join back
    return " ".join(tokens)

# Apply cleaning
df['cleaned_text'] = df['Consumer complaint narrative'].apply(clean_text)

print("Text preprocessing completed!")

Text preprocessing completed!


In [6]:
# ================================
# 5. Feature Engineering
# ================================

# 5.1 Complaint Length
df['complaint_length'] = df['cleaned_text'].apply(lambda x: len(x.split()))

# Preview
df[['cleaned_text', 'complaint_length']].head()

,cleaned_text,complaint_length
0,minimum payment fortiva retail credit card mon...,87
1,identity stolen someone created fraudelunet ac...,40
2,im sending complaint inform credit bureau vict...,71
3,identity theft xxxxxxxx xxxx xxxx xxxx xxxx xx...,405
4,filed numerous complaints credit bureau state ...,98


In [8]:
# ================================
# 6. Sentiment Score (Simple)
# ================================

# Using TextBlob (install if needed)
!pip install textblob

from textblob import TextBlob

def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

df['sentiment_score'] = df['cleaned_text'].apply(get_sentiment)

# Categorize sentiment
def sentiment_label(score):
    if score > 0:
        return 'Positive'
    elif score < 0:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment'] = df['sentiment_score'].apply(sentiment_label)

df[['cleaned_text', 'sentiment_score', 'sentiment']].head()

,cleaned_text,sentiment_score,sentiment
0,minimum payment fortiva retail credit card mon...,-0.031250,Negative
1,identity stolen someone created fraudelunet ac...,-0.412500,Negative
2,im sending complaint inform credit bureau vict...,-0.025000,Negative
3,identity theft xxxxxxxx xxxx xxxx xxxx xxxx xx...,0.112599,Positive
4,filed numerous complaints credit bureau state ...,0.275000,Positive


In [9]:
# ================================
# 7. Resolution Delay Calculation
# ================================

# Convert to datetime
df['Date received'] = pd.to_datetime(df['Date received'], errors='coerce')
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'], errors='coerce')

# Calculate delay (in days)
df['resolution_delay'] = (df['Date sent to company'] - df['Date received']).dt.days

# Handle negative or missing values
df['resolution_delay'] = df['resolution_delay'].apply(lambda x: x if x >= 0 else np.nan)

# Check results
df[['Date received', 'Date sent to company', 'resolution_delay']].head()

,Date received,Date sent to company,resolution_delay
0,2023-04-19,2023-04-19,0
1,2023-04-14,2023-04-14,0
2,2023-04-12,2023-04-12,0
3,2023-04-13,2023-04-13,0
4,2023-04-10,2023-04-10,0


In [10]:
# ================================
# 8. TF-IDF Vectorization
# ================================

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_tfidf = tfidf.fit_transform(df['cleaned_text'])

print("TF-IDF Shape:", X_tfidf.shape)

TF-IDF Shape: (1292265, 5000)


In [11]:
# ================================
# 9. Final Dataset Check
# ================================

print(df[['complaint_length', 'sentiment_score', 'resolution_delay']].describe())

       complaint_length  sentiment_score  resolution_delay
count      1.292265e+06     1.292265e+06      1.292265e+06
mean       9.367952e+01     2.556869e-02      1.172859e+00
std        1.138080e+02     1.701183e-01      7.850331e+00
min        0.000000e+00    -1.000000e+00      0.000000e+00
25%        3.300000e+01    -5.500000e-02      0.000000e+00
50%        6.200000e+01     0.000000e+00      0.000000e+00
75%        1.130000e+02     1.000000e-01      0.000000e+00
max        3.613000e+03     1.000000e+00      1.748000e+03
